# PDNO con simbolos parabolicos: laplaciano fraccionario 2D

Mismo experimento que `08_pdno_fraccionario_1d.ipynb` en $\mathbb T^2$ con la referencia $a(k)=|k|^\alpha$ (`FractionalLaplacianSymbol`, $n=2$) para $\alpha\in\{0.5,\ 1,\ 1.5\}$, aprendiendo con orden $m=\alpha$ ($\rho=n+1=3$), con los datos de `05_fraccionario_2d.ipynb`. Hay una seccion por $\alpha$ y, al final, una tabla resumen.

Arquitectura (PDNO, Shin et al., 2024, ec. (1)), simbolo par $a_{\text{par}}$, configuracion lineal y funcion de perdida: ver `06_pdno_calor_1d.ipynb`, introduccion. Aqui la red es el semigrupo $u(t)=e^{-t\bar a(D)}u_0$ con $\bar a=a_{\text{par}}$ ($L=1$, $\tau=1$).

# Setup
El codigo de cada seccion esta en `parabolic_symbol.learning.operator_experiment.PDNOExperiment`; cada metodo es una seccion de `06_pdno_calor_1d.ipynb`.

- **Datos:** los GRF de `05` en la malla de $128\times128$ puntos, $k_{\text{cap}}=32$, con particion **por condicion inicial**: la validacion son $u_0$ nuevas en `TRAIN_TIMES` (early stopping); la prueba, las $20$ $u_0$ de prueba en `TEST_TIMES`.
- **$\kappa$, $(c,C)$ y $C_\alpha$:** solo con las $u_0$ de entrenamiento (`ds.rows_train`), con $m=\alpha$ y la banda sobre $\langle k\rangle^\alpha$.
- **Entrenamiento y barrido:** `TRAIN_CONFIG` (valores por defecto de `OperatorTrainingConfig` salvo `patience=60`, con `class_C_source="data"` y $k_{\max}=k_{\text{cap}}$) y el barrido `SWEEP` de la MLP del simbolo, con la arquitectura del PDNO fija. Se elige la corrida de menor $L^2$ relativa de validacion.
- **Precision:** `float32` en todo (datos, modelo, entrenamiento, evaluacion, comprobaciones y modelo guardado). Los GRF se generan con numpy y se pasan a `float32` antes de que los vea el modelo.

La celda de parametros reune todo lo configurable.

In [ ]:
from pathlib import Path

import numpy as np
import torch

from parabolic_symbol.symbols.fractional_laplacian import FractionalLaplacianSymbol
from parabolic_symbol.learning.operator_experiment import PDNOExperiment, pdno_summary_figures, pdno_summary_table
from parabolic_symbol.learning.operator_training import OperatorTrainingConfig
from parabolic_symbol.utils import default_device

In [ ]:
# ---------------------------------------------------------------- parametros
# Precision y device: float32 en todo (datos, modelo, entrenamiento, barrido, evaluacion,
# comprobaciones y modelo guardado).
DEVICE = default_device()
DTYPE = torch.float32

# Datos: como en 05, una referencia |k|^alpha por alpha, con m = alpha.
ALPHAS = [0.5, 1.0, 1.5]
REFERENCES = {alpha: FractionalLaplacianSymbol(n=2, alpha=alpha) for alpha in ALPHAS}   # a(k) = |k|^alpha
M = {alpha: alpha for alpha in ALPHAS}                  # orden del simbolo aprendido: m = alpha
RESOLUTION = 128               # malla 128 x 128
K_CAP = 32                     # filas espectrales |k|_inf <= k_cap para (c, C) y C_alpha; k_max = k_cap
SEED = 0                       # 100 u0 de entrenamiento (TRAIN_TIMES) y 20 de prueba (TEST_TIMES)
VAL_FRACTION = 0.2             # fraccion de las u0 de entrenamiento que va a validacion
DELTA = 0.1                    # margen de la banda (c, C)
ESTIMATE_TOL = 1e-10           # filtro de senal de estimate_symbol

# Extrapolacion temporal: apagada. Ejemplo (t en [1, 2], todos > max(TRAIN_TIMES) = 0.95):
# EXTRAPOLATION_TIMES = np.linspace(1.0, 2.0, 11)
EXTRAPOLATION_TIMES = None

# Modelo: configuracion lineal del PDNO, la de 06. La MLP del simbolo y el weight decay salen del barrido.
ARCH = dict(width=1, use_W=False, bias=False, activation="identity", lift="identity", project="identity")
N_LAYERS = 1                   # L
LEARN_TAU = False              # tau fijo en 1/L
SYMBOL_HIDDEN = (64, 64, 64)   # solo para el modelo sin entrenar de las comprobaciones (seccion 3)
MODEL_SEED = 0

# Entrenamiento: valores por defecto de OperatorTrainingConfig salvo patience=60, con C_alpha de los datos.
TRAIN_CONFIG = OperatorTrainingConfig(k_max=K_CAP, class_C_source="data", patience=60)

# Barrido de la MLP del simbolo: el de 02-06 (SWEEP de learning/experiment.py), 30 corridas.
SWEEP = dict(
    depths=(2, 3, 4),
    widths=(32, 64, 128),
    dropout=0.2,
    dropout_layers=("none", "middle"),
    weight_decays=(0.0, 1e-4),
)

RESULTS = Path("../results/09_pdno_fraccionario_2d")   # tablas (CSV) y figuras (PNG); una subcarpeta por alpha y resumen.csv
RESULTS.mkdir(parents=True, exist_ok=True)
print(f"device = {DEVICE}  |  dtype = {DTYPE}")

In [ ]:
experimentos = {
    alpha: PDNOExperiment(
        reference=REFERENCES[alpha], m=M[alpha], resolution=RESOLUTION, k_cap=K_CAP,
        results=RESULTS / f"alpha_{alpha:g}", train_config=TRAIN_CONFIG, dtype=DTYPE,
        sweep_grid=SWEEP, device=DEVICE, seed=SEED, val_fraction=VAL_FRACTION, delta=DELTA,
        estimate_tol=ESTIMATE_TOL, extrapolation_times=EXTRAPOLATION_TIMES, arch=ARCH,
        n_layers=N_LAYERS, learn_tau=LEARN_TAU, symbol_hidden=SYMBOL_HIDDEN, model_seed=MODEL_SEED,
    )
    for alpha in ALPHAS
}

# $\alpha = 0.5$

$a(k)=|k|^{0.5}$, $m=0.5$, malla $128\times128$, $k_{\text{cap}}=32$. Tablas y figuras en `results/09_pdno_fraccionario_2d/alpha_0.5/`.

In [ ]:
exp_0_5 = experimentos[0.5]

## 1. Datos

`field_datasets` genera los mismos GRF que 03 / 05 y los devuelve como campos $u_0$ de forma `(n_ic, 128, 128)` y $u$ de forma `(n_ic, T, 128, 128)`, con todos los modos de la malla, en `DTYPE`. Particion por condicion inicial y comprobaciones de disjuncion: ver `06_pdno_calor_1d.ipynb`, seccion 1.

In [ ]:
display(exp_0_5.load_data())

**Condiciones iniciales y evolucion.** Dos $u_0$ de entrenamiento y la solucion exacta en algunos tiempos de `TRAIN_TIMES`, cada $u_0$ y cada tiempo en su imagen, con la misma escala de color.

In [ ]:
exp_0_5.initial_conditions()

## 2. Banda $(c,C)$, $\kappa$ y constantes $C_\alpha$

Como en 06, sobre `ds.rows_train` (las filas espectrales de las $u_0$ de entrenamiento): $\hat a(k)$, $(c,C)$ con $\delta=0.1$, $\kappa=$ `theoretical_kappa()` y las $C_\alpha$ con margen $1.5$ para todos los $\alpha$ con $|\alpha|\le\rho=n+1=3$ ($10$ multiindices en $\mathbb T^2$). Las del oraculo son solo comparacion.

In [ ]:
display(exp_0_5.estimate_constants())

## 3. Comprobaciones antes de entrenar

Con un modelo recien construido (pesos aleatorios, MLP `SYMBOL_HIDDEN`, modo eval), como en 06:

- **Paridad:** $\max_k|a_{\text{par}}(k)-a_{\text{par}}(-k)|$ en toda la rejilla $|k_i|\le64$ (Nyquist), incluidos los $k$ con $k_2=0$ y $k_2<0$: la costura del medio espacio de `rfftn` es toda la recta $k_2=0$. Debe ser $0$.
- **Reduccion:** el PDNO frente a $e^{-t\bar a(k)}\hat u_0(k)$ en el layout de los datos (`rfftn`) y, ademas, con la FFT completa (`fftn`, todos los modos, sin medio espacio). Error maximo relativo $\max_x|u_\theta-u|/\max_x|u|$.
- **Semigrupo:** $\max\|S(t+s)u_0-S(t)S(s)u_0\|/\|u_0\|$ para varios pares $(t,s)$.

Todo en `DTYPE`: reduccion y semigrupo deben quedar en unas pocas unidades del redondeo de float32 ($\sim10^{-6}$).

In [ ]:
display(exp_0_5.checks())

## 4. Barrido de la MLP del simbolo

El barrido de la celda de parametros (`SWEEP`, el de 02-06): $\{2,3,4\}$ capas ocultas $\times$ $\{32,64,128\}$ neuronas, dropout $0.2$ en `none` o `middle` y weight decay $\{0,10^{-4}\}$, $30$ corridas con la arquitectura del PDNO fija y `TRAIN_CONFIG`. Cada una es `train_operator` con `rows=ds.rows_train` (`class_C_source="data"`) y early stopping con paciencia $60$ sobre la perdida total de validacion. Detalles: ver `06_pdno_calor_1d.ipynb`, seccion 4.

In [ ]:
exp_0_5.sweep()

In [ ]:
barrido, por_eje = exp_0_5.sweep_results()
display(barrido)
for tabla in por_eje:
    display(tabla)

### Seleccion: la mejor aproximacion

La corrida de **menor $L^2$ relativa de validacion** (`best_run`). Desde aqui, las secciones 5 a 10 usan la corrida elegida.

In [ ]:
display(exp_0_5.select())

## 5. Curvas de entrenamiento

Perdida total y cada termino por epoca, en entrenamiento y validacion, cada uno en su imagen (`perdida_total.png`, `perdida_rel_l2.png`, `perdida_parabolicidad.png`, `perdida_clase.png`); la linea punteada marca la epoca restaurada por el early stopping. Las perdidas por epoca se guardan en `historia.csv`, que leen los benchmarks contra el FNO.

In [ ]:
exp_0_5.curves()

## 6. Prueba

`field_errors` sobre las $20$ $u_0$ de prueba en `TEST_TIMES`: error relativo $L^2$ en espacio fisico (malla $128\times128$) por condicion inicial y tiempo. Se imprime tambien la $L^2$ relativa sobre los pares $(k,t)$ de prueba con $|k|_\infty\le k_{\max}$, con el simbolo efectivo: la `test_rel_l2` de 05 y del resumen.

In [ ]:
display(exp_0_5.test())

**Campo.** Primera condicion inicial de prueba en algunos tiempos de `TEST_TIMES`: solucion exacta, la del PDNO y su diferencia, cada campo y tiempo en su imagen (`campo_exacta_t*.png`, `campo_pdno_t*.png`, `campo_error_t*.png`); exacta y PDNO comparten la escala de color.

In [ ]:
exp_0_5.field_figure()

## 7. Simbolo aprendido

El simbolo efectivo $\bar a(k)$ frente a $|k|^{0.5}$ en todos los modos de la malla, $|k_i|\le64$. El simbolo es radial, asi que se dibuja contra $|k|$; la ventana de entrenamiento es el cuadrado $|k|_\infty\le k_{\max}=32$, y los puntos dentro y fuera de ella van en colores distintos. En `simbolo_banda.png`, $\bar a(k)/\langle k\rangle^m$, que esta siempre en la banda $(c,C)$ por construccion.

In [ ]:
exp_0_5.symbol_figure()

**Error del simbolo.** $|\bar a(k)-a(k)|/a(k)$ en cada $k$ ($k=0$ se omite), dentro y fuera de la ventana, contra $|k|$ (`error_simbolo.png`) y como mapa sobre $(k_1,k_2)$ (`error_simbolo_mapa.png`).

In [ ]:
display(exp_0_5.symbol_error())

**Multiplicador.** $e^{-t\bar a(k)}$ aprendido (punteado) frente a $e^{-t|k|^{0.5}}$ (continuo) sobre el eje $k=(k_1,0)$, $0\le k_1\le k_{\max}$; su diferencia absoluta va en `multiplicador_error.png`.

In [ ]:
exp_0_5.multiplier()

## 8. Certificacion

**Parabolicidad (Def. 2.3).** `ParabolicityVerifier` sobre el simbolo de cada capa y el efectivo, con la rejilla de `GRFExperiment`: $|k_i|\le K$, $K\in\{k_{\max},2k_{\max},3k_{\max}\}$, y `CERT_LAM`. Se cumple si el peor ratio $\hat\kappa/\kappa$ es $\le1$.

**Clase $S^{m,\rho}$ (Def. 2.1).** `SymbolClassVerifier` en ventanas crecientes; es evidencia en una ventana finita, no una prueba sobre todo $\mathbb Z^n$.

In [ ]:
display(exp_0_5.certification())

In [ ]:
display(exp_0_5.symbol_class())

## 9. Extrapolacion temporal

Solo si `EXTRAPOLATION_TIMES` no es `None`: las $u_0$ de prueba en tiempos $t>0.95$. Solo se reporta.

In [ ]:
tabla_ext = exp_0_5.extrapolation()
if tabla_ext is not None:
    display(tabla_ext)

## 10. Modelo guardado

`save_operator_model` guarda en `results/09_pdno_fraccionario_2d/alpha_0.5/modelo.pt` la corrida seleccionada (pesos, `PDNOConfig`, `OperatorTrainingConfig`, $C_\alpha$, dtype y lo necesario para regenerar los datos). Para reconstruirlo sin este notebook:

```python
from parabolic_symbol.learning.operator_training import load_operator_datasets, load_operator_model
model = load_operator_model("results/09_pdno_fraccionario_2d/alpha_0.5/modelo.pt")
ds = load_operator_datasets("results/09_pdno_fraccionario_2d/alpha_0.5/modelo.pt", dtype=model.dtype)   # misma semilla, mismos datos
```

La ultima celda recarga el modelo y comprueba que da el mismo error de prueba.

In [ ]:
exp_0_5.save_model()

In [ ]:
exp_0_5.check_saved_model()

# $\alpha = 1$

$a(k)=|k|^{1}$, $m=1$, malla $128\times128$, $k_{\text{cap}}=32$. Tablas y figuras en `results/09_pdno_fraccionario_2d/alpha_1/`.

In [ ]:
exp_1_0 = experimentos[1.0]

## 1. Datos

In [ ]:
display(exp_1_0.load_data())

**Condiciones iniciales y evolucion.**

In [ ]:
exp_1_0.initial_conditions()

## 2. Banda $(c,C)$, $\kappa$ y constantes $C_\alpha$

In [ ]:
display(exp_1_0.estimate_constants())

## 3. Comprobaciones antes de entrenar

In [ ]:
display(exp_1_0.checks())

## 4. Barrido de la MLP del simbolo

In [ ]:
exp_1_0.sweep()

In [ ]:
barrido, por_eje = exp_1_0.sweep_results()
display(barrido)
for tabla in por_eje:
    display(tabla)

### Seleccion: la mejor aproximacion

In [ ]:
display(exp_1_0.select())

## 5. Curvas de entrenamiento

In [ ]:
exp_1_0.curves()

## 6. Prueba

In [ ]:
display(exp_1_0.test())

**Campo.**

In [ ]:
exp_1_0.field_figure()

## 7. Simbolo aprendido

In [ ]:
exp_1_0.symbol_figure()

**Error del simbolo.**

In [ ]:
display(exp_1_0.symbol_error())

**Multiplicador.**

In [ ]:
exp_1_0.multiplier()

## 8. Certificacion

In [ ]:
display(exp_1_0.certification())

In [ ]:
display(exp_1_0.symbol_class())

## 9. Extrapolacion temporal

In [ ]:
tabla_ext = exp_1_0.extrapolation()
if tabla_ext is not None:
    display(tabla_ext)

## 10. Modelo guardado

In [ ]:
exp_1_0.save_model()

In [ ]:
exp_1_0.check_saved_model()

# $\alpha = 1.5$

$a(k)=|k|^{1.5}$, $m=1.5$, malla $128\times128$, $k_{\text{cap}}=32$. Tablas y figuras en `results/09_pdno_fraccionario_2d/alpha_1.5/`.

In [ ]:
exp_1_5 = experimentos[1.5]

## 1. Datos

In [ ]:
display(exp_1_5.load_data())

**Condiciones iniciales y evolucion.**

In [ ]:
exp_1_5.initial_conditions()

## 2. Banda $(c,C)$, $\kappa$ y constantes $C_\alpha$

In [ ]:
display(exp_1_5.estimate_constants())

## 3. Comprobaciones antes de entrenar

In [ ]:
display(exp_1_5.checks())

## 4. Barrido de la MLP del simbolo

In [ ]:
exp_1_5.sweep()

In [ ]:
barrido, por_eje = exp_1_5.sweep_results()
display(barrido)
for tabla in por_eje:
    display(tabla)

### Seleccion: la mejor aproximacion

In [ ]:
display(exp_1_5.select())

## 5. Curvas de entrenamiento

In [ ]:
exp_1_5.curves()

## 6. Prueba

In [ ]:
display(exp_1_5.test())

**Campo.**

In [ ]:
exp_1_5.field_figure()

## 7. Simbolo aprendido

In [ ]:
exp_1_5.symbol_figure()

**Error del simbolo.**

In [ ]:
display(exp_1_5.symbol_error())

**Multiplicador.**

In [ ]:
exp_1_5.multiplier()

## 8. Certificacion

In [ ]:
display(exp_1_5.certification())

In [ ]:
display(exp_1_5.symbol_class())

## 9. Extrapolacion temporal

In [ ]:
tabla_ext = exp_1_5.extrapolation()
if tabla_ext is not None:
    display(tabla_ext)

## 10. Modelo guardado

In [ ]:
exp_1_5.save_model()

In [ ]:
exp_1_5.check_saved_model()

# Resumen

Una fila por $\alpha$, con las columnas de `results/05_fraccionario_2d/resumen.csv`: ventana de $k$, $c$, $C$, $\kappa$, configuracion elegida, $L^2$ relativa de validacion, error en prueba (pares $(k,t)$ con el simbolo efectivo y espacio fisico), error relativo medio del simbolo dentro de la ventana, peor ratio de la certificacion y multiindices consistentes del simbolo efectivo. Ademas, las del PDNO: error maximo en prueba en espacio fisico, error relativo medio del simbolo fuera de la ventana y las comprobaciones de paridad y semigrupo. Se guarda en `resumen.csv`.

In [ ]:
display(pdno_summary_table(experimentos, RESULTS))

Error en prueba (pares $(k,t)$ y espacio fisico) y peor ratio de la certificacion contra $\alpha$, leidos de `resumen.csv`, en dos imagenes: `resumen_error_prueba.png` y `resumen_certificacion.png` (la linea discontinua es la cota de la Def. 2.3, ratio $=1$).

In [ ]:
pdno_summary_figures(RESULTS / "resumen.csv", RESULTS)